# Foundations Assessment Data Mungling

## Tasks
1. Read in data to file.
2. Tidy up data.
3. Do some statistics.

In [2]:
import pandas as pd
import numpy as np
import os
import janitor
from pathlib import Path
import datetime as dt

In [2]:
os.chdir("work/")

FileNotFoundError: [Errno 2] No such file or directory: 'work/'

In [3]:
print(Path.cwd())

/home/jovyan/work/Assessment/Foundations_Assessment_Podfather


In [ ]:
# Read in listings csv
columns = ["id", "name", "description", "host_id", "host_name", "host_about", "host_location",
                         "host_total_listings_count", "host_is_superhost", "neighbourhood_cleansed",
                         "latitude", "longitude", "room_type", "bedrooms", "price", "minimum_nights",
                         "maximum_nights", "calendar_last_scraped", "number_of_reviews",
                         "estimated_occupancy_l365d", "estimated_revenue_l365d", "review_scores_rating", "availability_365",
                         "reviews_per_month", "first_review"]
listings = pd.read_csv("data/20260619-London-listings.csv",
                       usecols=columns,
                       dtype={'host_id': str, 'id': str})

# Read in calendar data
#calendar_data = pd.read_csv("data/calendar.csv")

# Reac in reviews data
reviews_data = pd.read_csv("data/reviews.csv")

OSError: [Errno 12] Cannot allocate memory

In [3]:
print(listings)

                        id                                               name  \
0                   264776                        Huge Four Bedroom Apartment   
1                   264777                              One Bedroom Apartment   
2                   264778            Two Bedroom Newly Refurbished Apartment   
3                   264779                  Refurbished Two Bedroom Apartment   
4                   264780    Spacious refurbished 2 bedroom apt with balcony   
...                    ...                                                ...   
96646  1439537980917449228                                 flat in shoreditch   
96647  1439588527887649898  Bright & Modern Studio in Dalston with a Patio E8   
96648  1439655439747195716       Modern 2 Bed 2bth in central Heart of London   
96649  1439664697712208758                      Chic apt close to Westminster   
96650  1439673341408524186            Spacious 2 bedroom flat in Notting Hill   

                           

In [5]:
print(listings.columns)
cols_to_covert = ["host_total_listings_count", "bedrooms", "price", "minimum_nights",
                  "maximum_nights", "number_of_reviews", "estimated_occupancy_l365d", "estimated_revenue_l365d",
                  "review_scores_rating", "reviews_per_month"]
datetime_cols = ["calendar_last_scraped", "first_review"]

listings[cols_to_covert] = listings[cols_to_covert].apply(lambda col: pd.to_numeric(col.astype(str).str.replace("$", "", regex=False), errors="coerce"))
listings[datetime_cols] = listings[datetime_cols].apply(lambda col: pd.to_datetime(col))

Index(['id', 'name', 'description', 'host_id', 'host_name', 'host_location',
       'host_about', 'host_is_superhost', 'host_total_listings_count',
       'neighbourhood_cleansed', 'latitude', 'longitude', 'room_type',
       'bedrooms', 'price', 'minimum_nights', 'maximum_nights',
       'calendar_last_scraped', 'number_of_reviews',
       'estimated_occupancy_l365d', 'estimated_revenue_l365d', 'first_review',
       'review_scores_rating', 'reviews_per_month'],
      dtype='str')


In [7]:
print(listings.dtypes)

id                                      str
name                                    str
description                             str
host_id                                 str
host_name                               str
host_location                           str
host_about                              str
host_is_superhost                       str
host_total_listings_count           float64
neighbourhood_cleansed                  str
latitude                            float64
longitude                           float64
room_type                               str
bedrooms                            float64
price                               float64
minimum_nights                        int64
maximum_nights                        int64
calendar_last_scraped        datetime64[us]
number_of_reviews                     int64
estimated_occupancy_l365d             int64
estimated_revenue_l365d             float64
first_review                 datetime64[us]
review_scores_rating            

In [6]:
# Filter to only lisitngs in london
london_listings = listings[(listings["host_location"].str.contains("London", case=False, regex=False)) &
                           (listings["host_location"].str.contains("United Kingdom", case=False, regex=False))]

# Filter to only listings of entire homes.
london_entire_homes = london_listings[london_listings["room_type"] == "Entire home/apt"].reset_index()

print(len(london_entire_homes["id"]))
print(len(london_listings["id"]))
print(london_entire_homes['price'].dtype)

34585
53239
float64


In [7]:
# Count the number of listings per host and get some stats on the price and bedroom counts.
listings_per_host_entire_homes = london_entire_homes.groupby("host_id").agg(
    host_name = ('host_name', 'first'),
    number_of_listings = ('id', 'count'),
    average_bedroom_count = ('bedrooms', 'mean'),
    price_median = ('price', 'median')
).round({'average_bedroom_count': 1, "average_price": 2})

total_listings_entire_homes = listings_per_host_entire_homes["number_of_listings"].sum() # Total listings with room types as entire homes.

listings_per_host_entire_homes["share_of_listings_%"] = (listings_per_host_entire_homes["number_of_listings"] / total_listings_entire_homes) * 100

print(total_listings_entire_homes)
print(listings_per_host_entire_homes.sort_values("number_of_listings", ascending=False).head(10))
print(listings_per_host_entire_homes.sort_values("share_of_listings_%", ascending=False).head(10).sum())


34585
             host_name  number_of_listings  average_bedroom_count  \
host_id                                                             
314162972   Blueground                 420                    1.4   
28820321   Tobias Mead                 282                    2.0   
1432477     City Relay                 246                    1.7   
33889201       Matthew                 197                    2.8   
47609036           Ben                 142                    1.3   
228928499      Beatrix                 129                    0.3   
124359784      Concept                 128                    1.0   
215357262         Alex                 115                    1.6   
83740964           Ali                 115                    1.4   
30253178        Alisha                 113                    1.5   

           price_median  share_of_listings_%  
host_id                                       
314162972         287.0             1.214399  
28820321          191.0 

In [8]:
# Get listing count per borough. 
borough_data = london_entire_homes.groupby("neighbourhood_cleansed").agg(
    price_median = ('price', 'median'),
    listing_count = ('id', 'count')
)

borough_data["share_of_total_%"] = (borough_data["listing_count"] / total_listings_entire_homes) * 100

print(borough_data.sort_values('share_of_total_%', ascending=False))

                        price_median  listing_count  share_of_total_%
neighbourhood_cleansed                                               
Westminster                    251.0           5033         14.552552
Kensington and Chelsea         249.0           3012          8.708978
Hackney                        150.0           2689          7.775047
Camden                         207.5           2568          7.425184
Tower Hamlets                  161.0           2483          7.179413
Islington                      165.0           2055          5.941882
Lambeth                        163.5           1938          5.603585
Southwark                      162.0           1930          5.580454
Wandsworth                     186.5           1888          5.459014
Hammersmith and Fulham         175.0           1677          4.848923
Brent                          157.0            965          2.790227
Haringey                       147.0            919          2.657221
Lewisham            

Lets take a look at all listings and get an idea of how many hosts own a large number of listings.

In [9]:
# Grouping the data by host and counting the number of listings each host has. 
# Also creating bins to categorize the hosts based on how many listings they have.
london_host_data = london_listings.groupby('host_id')['id'].count().reset_index().rename(columns={'id': 'listing_count'})
london_host_data['bin'] = pd.cut(london_host_data['listing_count'], [0, 1, 4, 19, 1000000000], labels=['Single', '2-4', '5-19', '20+']) # Add a column that divides the data into 4 bins depending on how many listings each host has.
total_listings = len(london_listings['id'])
print(total_listings)

# Sorting the host data to find the hosts with most listings.
print(london_host_data.sort_values(by='listing_count', ascending=False))
print(london_host_data.groupby("bin")['listing_count'].sum() / total_listings)

# Getting the share of hosts that are in each bin.
print(london_host_data["bin"].value_counts(normalize=True))

53239
         host_id  listing_count     bin
14011  314162972            420     20+
12869   28820321            285     20+
4046     1432477            246     20+
14981   33889201            197     20+
20113   47609036            142     20+
...          ...            ...     ...
14062  315576239              1  Single
14060    3155507              1  Single
14059   31550097              1  Single
14058    3154748              1  Single
15514   35294194              1  Single

[31028 rows x 3 columns]
bin
Single    0.469092
2-4       0.218562
5-19      0.153835
20+       0.158512
Name: listing_count, dtype: float64
bin
Single    0.804886
2-4       0.157567
5-19      0.031649
20+       0.005898
Name: proportion, dtype: float64


In [ ]:
# Getting the host data for all of the UK, not just london listings.
uk_host_data = listings.groupby('host_id')['id'].count().reset_index().rename(columns={'id': 'listing_count'})
print(uk_host_data.sort_values('listing_count', ascending=False))

uk_host_data["bin"] = pd.cut(uk_host_data['listing_count'], [0, 1, 4, 19, 1000000000], labels=['Single', '2-4', '5-19', '20+']) # Add a column that divides the data into 4 bins depending on how many listings each host has.

print(uk_host_data.groupby('bin')['listing_count'].sum())
print(uk_host_data['bin'].value_counts())



         host_id  listing_count
33255  446820235            495
24200  314162972            420
22143   28820321            285
6462     1432477            246
8261   156158778            213
...          ...            ...
20395  268065602              1
20396   26806796              1
20398   26809579              1
20399   26809748              1
55803   99999568              1

[55804 rows x 2 columns]
bin
Single    44976
2-4       20622
5-19      16233
20+       14820
Name: listing_count, dtype: int64
bin
Single    44976
2-4        8532
5-19       1957
20+         339
Name: count, dtype: int64


In [ ]:
# Getting a count for each room type (i.e. private room, or entire home/apt) in London.
room_type_count = london_listings["room_type"].value_counts()
print(room_type_count)

# Grouping the listings by host and getting some stats for the listings.
listings_per_host = london_listings.groupby("host_id").agg(
    host_name = ('host_name', 'first'),
    host_description = ('host_about', 'first'),
    listing_count = ('id', 'count'),
    average_bedroom_count = ('bedrooms', 'mean'),
    price_median = ('price', 'median')
).round({'average_bedroom_count': 1, "average_price": 2})

print(listings_per_host.sort_values('listing_count', ascending=False))

room_type
Entire home/apt    34585
Private room       18517
Shared room           79
Hotel room            58
Name: count, dtype: int64
             host_name                                   host_description  \
host_id                                                                     
314162972   Blueground  We’re Blueground, a global proptech company wi...   
28820321   Tobias Mead  I moved to London to follow my love who is a L...   
1432477     City Relay  City Relay is a property management company th...   
33889201       Matthew  Veeve offers the largest collection of handpic...   
47609036           Ben  I’m Ben from Croft Living. Croft Living is a f...   
...                ...                                                ...   
315576239        Seref                                                NaN   
3155507         Duncan  My names Duncan. I live in London and work as ...   
31550097     Christian                                                NaN   
3154748          

Next steps could be to compare the number of airbnbs in London to the total housing stock in London.